# Grokfast experiments

CSCI 567 final project: Vishal Patel, Salman Sattar, Tomohiko Uemura, Chinenye Okoro.

This notebook runs and analyzes every experiment in our study of
[*Grokfast: Accelerated Grokking by Amplifying Slow Gradients*](https://arxiv.org/abs/2405.20233)
on the modular-multiplication task (x·y mod 97). Training itself is the paper's own code
(`main.py`, with the filters in `grokfast.py`); this notebook defines the experiment grids,
runs them in parallel on one GPU, and turns the results into tables and plots.

[Open in Colab](https://colab.research.google.com/github/vip11017/csci567-grokfast/blob/salman/notebooks/grokfast_experiments.ipynb)
· use a **GPU runtime (T4 is enough)**: *Runtime → Change runtime type*.

| Section | What it does |
|---|---|
| 1. Setup | Clone the repo, check the GPU, choose where results are saved |
| 2. Experiment grids | Every configuration we run, by experiment |
| 3. Running grids | Background runner: `launch`, `status` |
| 4. Reading results | Load runs, steps to 95% validation accuracy, plots |
| 5. Replication | Baseline vs. Grokfast-MA vs. Grokfast-EMA, 3 seeds |
| 6. H2: weight decay | Does tuned weight decay alone close the gap? |
| 7. H1: momentum and frequency | Is it ordinary momentum, or the *slow* part of the gradient? |
| 8. Saving results | Keep results after the runtime shuts down |

Background and write-ups live in `docs/` (start with `docs/ideas.md` and `docs/results-replication.md`).

## 1. Setup

Clones the `salman` branch (or pulls it if it's already here) and checks the GPU.

In [ ]:
import base64
import glob
import gzip
import json
import os
import re
import statistics
import subprocess
import sys
import threading
import time

import matplotlib.pyplot as plt
import torch

REPO_URL = "https://github.com/vip11017/csci567-grokfast.git"
BRANCH = "salman"
REPO_DIR = "/content/csci567-grokfast"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

### Where results go

The Colab runtime is wiped when it shuts down. Set `USE_DRIVE = True` to write results
straight to Google Drive (Colab asks for permission once). Otherwise results stay in
`/content/results` until you save them (section 8).

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/csci567-grokfast"
else:
    RESULTS = "/content/results"
os.makedirs(RESULTS, exist_ok=True)
print("results go to", RESULTS)

## 2. Experiment grids

Each grid is a list of `main.py` runs. Every run stops once validation accuracy has held at
or above 95% for 1,000 steps, or at the step budget. Runs whose training accuracy is still
below 50% at step 20,000 are stopped and counted as failures: with strong weight decay
under Adam the model can't even fit the training set, and those runs would otherwise use
the whole budget.

Runs are ordered seed by seed, so a partly finished grid still covers every configuration
for seed 0.

In [ ]:
STOPPING = ["--stop_threshold", "0.95", "--stop_patience", "1000",
            "--abort_step", "20000", "--abort_train_acc", "0.5"]
MA = ["--filter", "ma", "--window_size", "100", "--lamb", "5.0"]
EMA = ["--alpha", "0.98", "--lamb", "2.0"]  # with --filter ema / ema_hp / ema_rand


def replication_grid(seed):
    """The paper's Table 3: baseline, Grokfast-MA, Grokfast-EMA."""
    yield f"rep_s{seed}_none", f"rep_s{seed}", []
    yield f"rep_s{seed}_ma", f"rep_s{seed}", MA + ["--weight_decay", "0.01"]
    yield f"rep_s{seed}_ema", f"rep_s{seed}", ["--filter", "ema"] + EMA + ["--weight_decay", "0.005"]


def h2_grid(seed):
    """Optimizer x weight decay x filter. Skips AdamW at 0 (same as Adam) and Adam/none/0
    (the replication baseline)."""
    for optimizer in ("Adam", "AdamW"):
        for filt in ("none", "ema"):  # slow no-filter runs first, for better packing
            for wd in (0, 0.001, 0.005, 0.01, 0.05, 0.1, 1.0):
                if wd == 0 and (optimizer == "AdamW" or filt == "none"):
                    continue
                args = ["--optimizer", optimizer, "--weight_decay", str(wd)]
                if filt == "ema":
                    args += ["--filter", "ema"] + EMA
                yield f"h2_s{seed}_{optimizer.lower()}_{filt}_wd{wd:g}", f"h2_s{seed}_{optimizer.lower()}", args


def h1_grid(seed):
    """Filters at the same gain, and more Adam momentum without a filter."""
    for wd in (0, 0.005):
        for filt in ("none", "ema", "ema_hp", "ema_rand"):
            args = ["--weight_decay", str(wd)]
            if filt != "none":
                args += ["--filter", filt] + EMA
            yield f"h1_s{seed}_b0.9_{filt}_wd{wd:g}", f"h1_s{seed}_b0.9", args
        for beta1 in (0.95, 0.98, 0.99):
            yield (f"h1_s{seed}_b{beta1}_none_wd{wd:g}", f"h1_s{seed}_b{beta1}",
                   ["--weight_decay", str(wd), "--beta1", str(beta1)])


GRIDS = {"replication": replication_grid, "h2": h2_grid, "h1": h1_grid}


def build_runs(grid, seeds=(0, 1, 2), budget=100000):
    out = f"{RESULTS}/{grid}"
    runs = []
    for seed in seeds:
        for name, label, args in GRIDS[grid](seed):
            cmd = [sys.executable, "main.py", "--label", label, "--seed", str(seed),
                   "--budget", str(budget), "--out_dir", out] + STOPPING + args
            runs.append({"name": name, "cmd": cmd})
    return runs


for grid in GRIDS:
    print(f"{grid}: {len(build_runs(grid))} runs")

## 3. Running grids

`launch(grid)` runs a grid in a background thread, a fixed number of `main.py` processes at a
time, so the cell returns right away. On a T4, 8 at once works well: the model is tiny, so
most of the cost is Python overhead, not the GPU. A Grokfast run takes a few minutes; a
no-filter baseline can take an hour.

Each finished run leaves a `done_<name>.txt` marker and each started run a `pid_<name>.txt`,
so calling `launch` again (say, after a kernel restart) skips finished runs and picks up runs
that are still going instead of starting them twice. `status(grid)` shows progress.

In [ ]:
RUNNERS = {}


def _alive(pid):
    return os.path.exists(f"/proc/{pid}")


def _run_grid(grid, runs, parallel):
    out = f"{RESULTS}/{grid}"
    done = lambda name: os.path.exists(f"{out}/done_{name}.txt")
    queue = [r for r in runs if not done(r["name"])]
    running = {}
    for run in list(queue):  # adopt runs still going from an earlier kernel
        pid_file = f"{out}/pid_{run['name']}.txt"
        if os.path.exists(pid_file) and _alive(int(open(pid_file).read())):
            running[run["name"]] = int(open(pid_file).read())
            queue.remove(run)

    while queue or running:
        while queue and len(running) < parallel:
            run = queue.pop(0)
            with open(f"{out}/log_{run['name']}.txt", "w") as log:
                proc = subprocess.Popen(run["cmd"], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT,
                                        stdin=subprocess.DEVNULL, start_new_session=True)
            with open(f"{out}/pid_{run['name']}.txt", "w") as f:
                f.write(str(proc.pid))
            running[run["name"]] = proc

        for name, proc in list(running.items()):
            if isinstance(proc, subprocess.Popen):
                finished, rc = proc.poll() is not None, proc.returncode
            else:
                finished, rc = not _alive(proc), "unknown"
            if finished:
                with open(f"{out}/done_{name}.txt", "w") as f:
                    f.write(f"rc={rc}\n")
                del running[name]

        with open(f"{out}/status.json", "w") as f:
            json.dump({"grid": grid, "updated": time.strftime("%Y-%m-%d %H:%M:%S"),
                       "total": len(runs), "done": sum(done(r["name"]) for r in runs),
                       "running": sorted(running), "queued": [r["name"] for r in queue]}, f, indent=1)
        time.sleep(10)


def launch(grid, parallel=8, seeds=(0, 1, 2), budget=100000):
    """Start (or resume) a grid in the background."""
    if grid in RUNNERS and RUNNERS[grid].is_alive():
        print(f"{grid} is already running")
        return
    os.makedirs(f"{RESULTS}/{grid}", exist_ok=True)
    runs = build_runs(grid, seeds, budget)
    RUNNERS[grid] = threading.Thread(target=_run_grid, args=(grid, runs, parallel), daemon=True)
    RUNNERS[grid].start()
    print(f"launched {grid}: {len(runs)} runs -> {RESULTS}/{grid}")


def status(grid):
    """How many runs are done, running and queued."""
    with open(f"{RESULTS}/{grid}/status.json") as f:
        s = json.load(f)
    print(f"{grid}: {s['done']}/{s['total']} done, {len(s['running'])} running, "
          f"{len(s['queued'])} queued (updated {s['updated']} UTC)")
    for name in s["running"]:
        print("  running:", name)

## 4. Reading results

`main.py` saves each run as `res_<name>.pt` with its training and validation curves (one point
every 10 steps). The main number we report is the **first step with validation accuracy
≥ 95%**, as in the paper.

In [ ]:
THRESHOLD = 0.95


def first_step_at(its, accs, threshold=THRESHOLD):
    return next((it for it, acc in zip(its, accs) if acc >= threshold), None)


def load_runs(folder):
    """All res_*.pt files in a folder, keyed by run name."""
    runs = {}
    for path in sorted(glob.glob(f"{folder}/res_*.pt")):
        r = torch.load(path, weights_only=True)
        r["first_95"] = first_step_at(r["its"], r["val_acc"])
        runs[os.path.basename(path)[4:-3]] = r
    return runs


def summary(folder, match=""):
    """One line per run."""
    for name, r in load_runs(folder).items():
        if match in name:
            print(f"{name:45s} first95={r['first_95']}  last={r['its'][-1]}  val={r['val_acc'][-1]:.3f}")


def by_group(runs):
    """Group runs by everything after the seed tag: rep_s0_none -> 'none'."""
    groups = {}
    for name, r in runs.items():
        m = re.search(r"_s(\d+)_(.+)$", name)
        if m:
            groups.setdefault(m.group(2), {})[int(m.group(1))] = r
    return groups


def cell(seed_runs):
    """Median steps to 95% over seeds that reached it, with per-seed values."""
    hits = {s: r["first_95"] for s, r in sorted(seed_runs.items())}
    reached = [h for h in hits.values() if h is not None]
    per_seed = "/".join("-" if h is None else f"{h / 1000:.1f}k" for h in hits.values())
    median = f"{statistics.median(reached) / 1000:.1f}k" if 2 * len(reached) > len(hits) else "fail"
    return f"{median} ({per_seed})"


def plot_val_acc(runs, title, groups=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    for color, (group, seed_runs) in zip(colors, sorted(by_group(runs).items())):
        if groups and group not in groups:
            continue
        for i, r in enumerate(seed_runs.values()):
            ax.plot(r["its"], r["val_acc"], color=color, lw=1, alpha=0.8, label=group if i == 0 else None)
    ax.axhline(THRESHOLD, color="grey", ls="--", lw=0.8)
    ax.set(xscale="log", xlabel="Optimization steps", ylabel="Validation accuracy", title=title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.show()

## 5. Replication

The paper's headline comparison (Table 3): baseline Adam with no weight decay, Grokfast-MA
(window 100, λ 5, weight decay 0.01) and Grokfast-EMA (α 0.98, λ 2, weight decay 0.005).
Paper: 39,890 / 790 / 910 steps to 95% validation accuracy.

We ran this on 2026-10-30 (3 seeds, Colab T4); the saved runs are in `results/replication/`
and the write-up is `docs/results-replication.md`. To rerun it, uncomment `launch` (about
1.5 hours on a T4, mostly the baseline runs) and point `folder` at `f"{RESULTS}/replication"`.

In [ ]:
# launch("replication")
folder = f"{REPO_DIR}/results/replication"
replication = load_runs(folder)
for group, seed_runs in sorted(by_group(replication).items()):
    print(f"{group:25s} {cell(seed_runs)}")
plot_val_acc(replication, "Replication, 3 seeds")

## 6. H2: weight decay vs. the filter

The paper's baseline uses no weight decay, while its Grokfast runs use 0.005–0.01. This grid
asks whether tuned weight decay alone explains most of the speedup:

- optimizer: Adam (weight decay added to the gradient, as in the paper's `main.py`) and AdamW (decoupled)
- weight decay: 0, 0.001, 0.005, 0.01, 0.05, 0.1, 1.0
- filter: none, Grokfast-EMA (α 0.98, λ 2)
- seeds 0–2, 75 runs, about 5 hours on a T4

Note that Grokfast amplifies the loss gradient *before* Adam adds the weight-decay term, so at
the same nominal weight decay the filter makes decay relatively weaker.

In [ ]:
launch("h2")

In [ ]:
status("h2")

In [ ]:
h2 = load_runs(f"{RESULTS}/h2")
groups = by_group(h2)
groups["adam_none"] = by_group(replication)["none"]  # weight decay 0 baseline, from section 5

columns = [("adam_none", "Adam, no filter"), ("adam_ema_a0980_l2", "Adam + EMA"),
           ("adamw_none", "AdamW, no filter"), ("adamw_ema_a0980_l2", "AdamW + EMA")]
suffix = {0: "", 0.001: "_wd10e-03", 0.005: "_wd50e-03", 0.01: "_wd10e-02",
          0.05: "_wd50e-02", 0.1: "_wd10e-01", 1.0: "_wd10e+00"}

print("Steps to 95% val accuracy: median (seed 0/1/2)\n")
print(f"{'weight decay':>12} | " + " | ".join(f"{title:24s}" for _, title in columns))
for wd, s in suffix.items():
    row = []
    for key, _ in columns:
        if wd == 0 and key.startswith("adamw"):
            row.append("(same as Adam)")
        else:
            seed_runs = groups.get(key + s)
            row.append(cell(seed_runs) if seed_runs else "")
    print(f"{wd:>12g} | " + " | ".join(f"{c:24s}" for c in row))

## 7. H1: ordinary momentum, or the slow part of the gradient?

Adam already averages gradients (β₁ = 0.9); Grokfast-EMA adds a second, slower average in
front of it. This grid separates the two explanations, with Adam at weight decay 0 and 0.005:

- **filters at the same gain (λ 2, α 0.98):** none, EMA (Grokfast), high-pass EMA
  `g + λ(g − EMA(g))`, and a random direction with the same size as `λ·EMA(g)`
- **more Adam momentum, no filter:** β₁ ∈ {0.95, 0.98, 0.99}
- seeds 0–2, 42 runs

If the slow component is what matters, Grokfast should beat both the high-pass and random
filters, and higher β₁ shouldn't match it.

In [ ]:
launch("h1")

In [ ]:
status("h1")

In [ ]:
h1 = load_runs(f"{RESULTS}/h1")
for group, seed_runs in sorted(by_group(h1).items()):
    print(f"{group:35s} {cell(seed_runs)}")

## 8. Saving results

If `USE_DRIVE` was off, copy everything to Drive before the runtime shuts down:

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/csci567-grokfast && cp -r {RESULTS}/* /content/drive/MyDrive/csci567-grokfast/

Without Drive, `export(grid)` prints a compressed copy of every run's curves (downsampled to
at most 300 points) that can be pasted into a text file and turned back into `res_<name>.json`
files with `decode`.

In [ ]:
def export(grid, match="", max_points=300):
    runs = {}
    for name, r in load_runs(f"{RESULTS}/{grid}").items():
        if match not in name:
            continue
        step = max(1, len(r["its"]) // max_points)
        keep = sorted(set(range(0, len(r["its"]), step)) | {len(r["its"]) - 1})
        runs[name] = {"first_95": r["first_95"], **{k: [r[k][i] for i in keep]
                      for k in ("its", "train_acc", "val_acc", "train_loss", "val_loss")}}
    print(base64.b64encode(gzip.compress(json.dumps(runs).encode())).decode())


def decode(blob_file, out_dir):
    runs = json.loads(gzip.decompress(base64.b64decode(open(blob_file).read().strip())))
    os.makedirs(out_dir, exist_ok=True)
    for name, run in runs.items():
        with open(f"{out_dir}/res_{name}.json", "w") as f:
            json.dump(run, f)
    print(f"wrote {len(runs)} runs to {out_dir}")